# Machine Learning Assignment 2 - Resubmission

**Dataset**:       Fitness


## Imports

Add imports here as needed.

Remember to **re-run the cell when you add imports**, so it gets loaded into the virtual notebook environment!

In [1]:
# Data and Datasets
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

# Clustering
from sklearn.cluster import DBSCAN

# Validation methods
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold

# Metrics
from sklearn import metrics
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.neighbors import KNeighborsClassifier

# Hyper-parameter optimisation
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Feature selection & feature engineering
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from sklearn.decomposition import PCA

# Stats
from scipy.stats import randint as sp_randint
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import shapiro     # Shapiro Wilk
from scipy.stats import normaltest  # D’Agostino’s K^2
from scipy.stats import anderson    # Anderson-Darling
from scipy.stats import ttest_ind    # independent student t-test; assumes normality
from scipy.stats import mannwhitneyu # non-parametric; doesn't assume normality

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import SVG
from IPython.display import display
from sklearn.tree import export_graphviz

# Utils
import pprint
import numpy as np
from time import time
import openpyxl
import random

## Loading the dataset

In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Drin\OneDrive - Jakupi\Documents\ml_cwk_2_resub\data\fitness_dataset.csv')

# printing out dataset info
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1610 entries, 0 to 1609
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Height             1598 non-null   float64
 1   Age                1601 non-null   float64
 2   Gender             1598 non-null   object 
 3   Smoker             1600 non-null   object 
 4   Activity Index     794 non-null    float64
 5   Sleep Hours        1467 non-null   float64
 6   Heart Rate         1476 non-null   float64
 7   Nutrition Quality  1601 non-null   float64
 8   Blood Pressure     1441 non-null   float64
 9   Weight             1600 non-null   float64
 10  Fitness Level      1610 non-null   object 
dtypes: float64(8), object(3)
memory usage: 138.5+ KB
None


,Height,Age,Gender,Smoker,Activity Index,Sleep Hours,Heart Rate,Nutrition Quality,Blood Pressure,Weight,Fitness Level
0,152.0,56.0,Female,No,3.97,NaN,69.6,2.37,117.0,65.0,High
1,186.0,69.0,Female,No,3.19,7.5,60.8,8.77,114.8,95.0,High
2,192.0,46.0,Female,No,2.03,NaN,61.4,8.20,116.4,103.0,Low
3,189.0,32.0,Male,No,NaN,7.0,60.2,6.18,130.1,83.0,High
4,175.0,60.0,Female,Yes,NaN,8.0,58.1,9.95,115.8,99.0,High


In [5]:
df.describe()

,Height,Age,Activity Index,Sleep Hours,Heart Rate,Nutrition Quality,Blood Pressure,Weight
count,1598.000000,1601.000000,794.000000,1467.000000,1476.000000,1601.000000,1441.000000,1600.000000
mean,174.080100,50.531543,2.875982,7.431152,70.233469,4.846471,118.836572,84.207500
std,14.380676,17.870658,1.127762,1.499842,11.848808,2.851291,18.142572,26.162062
min,150.000000,18.000000,1.000000,4.000000,45.000000,0.000000,0.000000,30.000000
25%,162.000000,36.000000,1.910000,6.400000,62.375000,2.420000,109.200000,64.000000
50%,173.000000,51.000000,2.780000,7.400000,70.100000,4.730000,119.800000,84.000000
75%,186.000000,66.000000,3.780000,8.500000,78.300000,7.210000,130.000000,103.000000
max,199.000000,79.000000,4.990000,12.000000,118.600000,10.000000,168.100000,250.000000


## Data Processing

### Data Quality

TODO: fix data quality issues identified in your exploratory data analysis

In [6]:
df_clean = df.copy()
print(f"Starting shape: {df_clean.shape}")

Starting shape: (1610, 11)


In [7]:
bp_zero_count = (df_clean['Blood Pressure'] == 0).sum()
df_clean.loc[df_clean['Blood Pressure'] == 0, 'Blood Pressure'] = pd.NA
df_clean['Blood Pressure'] = pd.to_numeric(df_clean['Blood Pressure'])
print(f"Recoded {bp_zero_count} Blood Pressure values (0 -> NaN)")

Recoded 12 Blood Pressure values (0 -> NaN)


In [8]:
weight_noise_mask = (df_clean['Weight'] < 35) | (df_clean['Weight'] > 150)
weight_noise_count = weight_noise_mask.sum()
df_clean.loc[weight_noise_mask, 'Weight'] = pd.NA
df_clean['Weight'] = pd.to_numeric(df_clean['Weight'])
print(f"Recoded {weight_noise_count} extreme Weight values (>150 or <35) -> NaN")

Recoded 22 extreme Weight values (>150 or <35) -> NaN


In [9]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} duplicate row(s). New shape: {df_clean.shape}")

Dropped 1 duplicate row(s). New shape: (1609, 11)


In [10]:
df_clean['Activity_Index_was_missing'] = df_clean['Activity Index'].isna()
activity_median = df_clean['Activity Index'].median()
df_clean['Activity Index'] = df_clean['Activity Index'].fillna(activity_median)
print(f"Activity Index imputed with median = {activity_median}")
print(f"Flag column added: {df_clean['Activity_Index_was_missing'].sum()} rows flagged as originally missing")

Activity Index imputed with median = 2.78
Flag column added: 815 rows flagged as originally missing


**5. Remaining missing values — impute the rest**

For the remaining numeric columns we use median imputation (robust to the outliers/skew found earlier). For the categorical columns (Gender, Smoker) we use the mode.

In [11]:
numeric_to_impute = ['Height', 'Age', 'Sleep Hours', 'Heart Rate', 'Nutrition Quality', 'Blood Pressure', 'Weight']
categorical_to_impute = ['Gender', 'Smoker']

for col in numeric_to_impute:
    median_val = df_clean[col].median()
    n_missing = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna(median_val)
    if n_missing > 0:
        print(f"{col}: imputed {n_missing} missing value(s) with median = {median_val}")

for col in categorical_to_impute:
    mode_val = df_clean[col].mode()[0]
    n_missing = df_clean[col].isna().sum()
    df_clean[col] = df_clean[col].fillna(mode_val)
    if n_missing > 0:
        print(f"{col}: imputed {n_missing} missing value(s) with mode = '{mode_val}'")

Height: imputed 11 missing value(s) with median = 173.0
Age: imputed 8 missing value(s) with median = 51.0
Sleep Hours: imputed 142 missing value(s) with median = 7.4
Heart Rate: imputed 133 missing value(s) with median = 70.1
Nutrition Quality: imputed 8 missing value(s) with median = 4.73
Blood Pressure: imputed 180 missing value(s) with median = 119.9
Weight: imputed 31 missing value(s) with median = 83.0
Gender: imputed 11 missing value(s) with mode = 'Female'
Smoker: imputed 9 missing value(s) with mode = 'No'


In [12]:
# Final check — confirm no missing values remain
print("Remaining missing values per column:")
print(df_clean.isna().sum())
print(f"\nFinal cleaned shape: {df_clean.shape}")

Remaining missing values per column:
Height                        0
Age                           0
Gender                        0
Smoker                        0
Activity Index                0
Sleep Hours                   0
Heart Rate                    0
Nutrition Quality             0
Blood Pressure                0
Weight                        0
Fitness Level                 0
Activity_Index_was_missing    0
dtype: int64

Final cleaned shape: (1609, 12)


In [13]:
print(f"Shape before cleaning: {df.shape}")
print(f"Shape after cleaning:  {df_clean.shape}")
print(f"\nMissing values remaining:\n{df_clean.isna().sum().sum()} total")
print()
df_clean.describe().T

Shape before cleaning: (1610, 11)
Shape after cleaning:  (1609, 12)

Missing values remaining:
0 total



,count,mean,std,min,25%,50%,75%,max
Height,1609.0,174.072716,14.331681,150.0,162.00,173.00,186.00,199.00
Age,1609.0,50.533872,17.826178,18.0,36.00,51.00,66.00,79.00
Activity Index,1609.0,2.827365,0.793428,1.0,2.78,2.78,2.78,4.99
Sleep Hours,1609.0,7.428403,1.432115,4.0,6.60,7.40,8.30,12.00
Heart Rate,1609.0,70.222436,11.348277,45.0,63.10,70.10,77.40,118.60
Nutrition Quality,1609.0,4.845892,2.844201,0.0,2.44,4.73,7.20,10.00
Blood Pressure,1609.0,119.841827,13.729173,90.0,110.80,119.90,128.70,168.10
Weight,1609.0,82.830951,21.680774,35.0,65.00,83.00,102.00,119.00


### Data Enhancements

TODO: do data enhancements here, e.g.,
* Feature encoding
* Feature engineering (like PCA)
* Feature scaling
* Modifying class balance (though this should only be done on test data, so may need to be done within your ML validation code cell and not here)



In [14]:
# Encode categorical features (binary, so simple mapping is sufficient — no need for one-hot encoding overhead)
df_model = df_clean.copy()
df_model['Gender_enc'] = (df_model['Gender'] == 'Male').astype(int)
df_model['Smoker_enc'] = (df_model['Smoker'] == 'Yes').astype(int)
df_model['Activity_Index_was_missing'] = df_model['Activity_Index_was_missing'].astype(int)

# Encode the target: High = 1, Low = 0
df_model['Fitness_binary'] = (df_model['Fitness Level'] == 'High').astype(int)

feature_cols = ['Height', 'Age', 'Activity Index', 'Sleep Hours', 'Heart Rate',
                'Nutrition Quality', 'Blood Pressure', 'Weight',
                'Gender_enc', 'Smoker_enc', 'Activity_Index_was_missing']

X = df_model[feature_cols]
y = df_model['Fitness_binary']

print(f"Feature matrix shape: {X.shape}")
print(f"Target balance:\n{y.value_counts(normalize=True)}")

Feature matrix shape: (1609, 11)
Target balance:
Fitness_binary
0    0.747669
1    0.252331
Name: proportion, dtype: float64


**Design notes — data enhancement:**

- `Gender` and `Smoker` are both binary categorical features, so a simple 0/1 mapping is used rather than one-hot encoding (which would be needed for 3+ categories) — this keeps the feature set compact and avoids unnecessary dummy columns.
- `Activity_Index_was_missing` (created during cleaning in Assignment 1) is retained as a feature: since ~51% of Activity Index was imputed, whether a row was originally missing may itself carry information worth letting the models use.
- Feature scaling is applied later, inside the validation step, for the models that need it (Logistic Regression, KNN) rather than here — this avoids leaking test-set statistics into the scaler before the train/test split happens.

## ML Validation

Add code for validating your chosen ML models here.

OBS: some data processing may need to be done here too, depending on your workflow design.

### Task 2.1 — Workflow Design & Justification

**Models chosen (7) — broad coverage across model families:**

| Model | Family | Role |
|---|---|---|
| Logistic Regression | Linear | Interpretable baseline |
| Random Forest | Tree ensemble (bagging) | Non-linear interactions |
| Hist Gradient Boosting | Tree ensemble (boosting) | Strong tabular-data booster |
| K-Nearest Neighbors | Instance-based | Non-parametric contrast |
| Naive Bayes | Probabilistic | Simple, robust baseline on noisy data |
| SVM (RBF) | Margin-based | Non-linear decision boundary |
| MLP | Neural network | Simple 1-2 hidden layer network |

Seven models (up from an earlier 3-model version) gives broader coverage across fundamentally different modelling assumptions — linear, tree-based, instance-based, probabilistic, margin-based, and neural — rather than relying on a small, similar set.

**Class imbalance — SMOTE, not just `class_weight`:**

The target is ~75% Low / 25% High. A model that always predicts "Low" would score ~75% accuracy while finding zero "High" cases — accuracy alone is not a usable metric here. Rather than only reweighting classes (`class_weight='balanced'`, used in an earlier version), this version uses **SMOTE** (Synthetic Minority Oversampling) to generate synthetic "High" examples during training. SMOTE is placed *inside* the pipeline (via `imblearn`'s `Pipeline`) so it only ever touches the training folds — validation and test data never see synthetic samples, which avoids leaking information into the evaluation.

**Metric: macro-F1, not `f1` or accuracy.**

Plain `f1` scores only the positive ("High") class; macro-F1 averages F1 across *both* classes equally, so neither the majority nor minority class dominates the ranking. This is the single metric used consistently for both hyperparameter tuning and final model selection.

**Hyper-parameters: `RandomizedSearchCV` for every model, all scored on macro-F1.**

As before, every model is tuned — not just some — so no model is compared against another using arbitrary defaults. `RandomizedSearchCV` (rather than an exhaustive `GridSearchCV`) is used here since the combined search space across seven models is larger; each model draws up to 12 hyper-parameter combinations from its own grid, using the same `StratifiedKFold` cross-validation throughout.

**Validation and selection rule:**

- Stratified 80/20 train/test split, plus 5-fold `StratifiedKFold` used throughout tuning.
- **The winning model is selected by CV macro-F1 — not by peeking at the test set.** The test set is used only afterward, as a sanity check on the selected model, not as the selection criterion itself. If a model looks slightly better on the single test split than the CV-selected winner, that is treated as expected sampling variance, not a reason to switch — switching would mean implicitly fitting the model choice to the test set.

### Task 2.2 — Implementation

#### Pipelines with SMOTE

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, learning_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
N_ITER = 12

# Stratified train/test split — preserves the 75/25 class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train target balance:\n{y_train.value_counts(normalize=True)}")

In [ ]:
def make_pipe(clf, needs_scaling):
    """Build a pipeline: optional scaling -> SMOTE (train folds only) -> classifier."""
    steps = []
    if needs_scaling:
        steps.append(('scaler', StandardScaler()))
    steps.append(('smote', SMOTE(random_state=SEED)))
    steps.append(('clf', clf))
    return ImbPipeline(steps)

# (model, needs_scaling) — distance/gradient-based models need scaling, tree-based ones don't
candidate_models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=SEED), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=SEED), False),
    'Hist Gradient Boosting': (HistGradientBoostingClassifier(random_state=SEED), False),
    'KNN': (KNeighborsClassifier(), True),
    'Naive Bayes': (GaussianNB(), True),
    'SVM (RBF)': (SVC(kernel='rbf', random_state=SEED), True),
    'MLP': (MLPClassifier(hidden_layer_sizes=(64,), max_iter=400, early_stopping=True, random_state=SEED), True),
}

#### Baseline (untuned) comparison — diagnosing overfitting

Before any tuning, it's worth seeing how each model behaves with its library defaults — this motivates *why* tuning matters, with evidence, rather than just asserting it.

In [ ]:
baseline_rows = []
for name, (clf, needs_scaling) in candidate_models.items():
    pipe = make_pipe(clf, needs_scaling)
    pipe.fit(X_train, y_train)
    train_f1 = f1_score(y_train, pipe.predict(X_train), average='macro')
    test_f1 = f1_score(y_test, pipe.predict(X_test), average='macro')
    baseline_rows.append({
        'Model': name, 'Train macro-F1': train_f1, 'Test macro-F1': test_f1,
        'Gap': train_f1 - test_f1,
    })

baseline_df = pd.DataFrame(baseline_rows).round(4)
baseline_df

**Draft notes — rewrite in your own words for the video:**

- Random Forest and Hist Gradient Boosting hit near-perfect training F1 (close to 1.0) with default settings — textbook overfitting/memorisation, not genuine learning. Their train-test gap is by far the largest of the seven models.
- Logistic Regression, Naive Bayes, and MLP show small gaps even untuned — simpler models are naturally less prone to memorising this dataset.
- This is direct evidence for why every model needs tuning, not just an assumption: without it, the tree-based ensembles would look artificially strong on paper while actually being poorly regularised.

#### Hyperparameter tuning — RandomizedSearchCV for every model

In [ ]:
search_spaces = {
    'Logistic Regression': {'clf__C': [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 10]},
    'Random Forest': {
        'clf__max_depth': [3, 4, 5, 8],
        'clf__min_samples_leaf': [1, 2, 4, 10, 20],
        'clf__n_estimators': [100, 200, 300],
    },
    'Hist Gradient Boosting': {
        'clf__max_depth': [2, 3, None],
        'clf__learning_rate': [0.02, 0.05, 0.1],
        'clf__max_iter': [50, 100, 150],
        'clf__min_samples_leaf': [10, 20, 40],
    },
    'KNN': {
        'clf__n_neighbors': [5, 10, 15, 20, 25, 31, 41],
        'clf__weights': ['uniform', 'distance'],
    },
    'Naive Bayes': {'clf__var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]},
    'SVM (RBF)': {'clf__C': [0.1, 1.0, 10.0], 'clf__gamma': ['scale', 0.01, 0.1]},
    'MLP': {
        'clf__hidden_layer_sizes': [(32,), (64,), (64, 32)],
        'clf__alpha': [1e-4, 1e-3, 1e-2],
    },
}

best_estimators = {}
tuning_rows = []
for name, (clf, needs_scaling) in candidate_models.items():
    pipe = make_pipe(clf, needs_scaling)
    params = search_spaces[name]
    n_combos = 1
    for v in params.values():
        n_combos *= len(v)
    n_iter = min(N_ITER, n_combos)

    search = RandomizedSearchCV(
        pipe, params, n_iter=n_iter, cv=cv, scoring='f1_macro',
        random_state=SEED, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    best_estimators[name] = search.best_estimator_
    print(f"{name}: best_params={search.best_params_}, CV macro-F1={search.best_score_:.4f}")
    tuning_rows.append({'Model': name, 'CV macro-F1': search.best_score_, 'Best Params': str(search.best_params_)})

tuning_df = pd.DataFrame(tuning_rows).sort_values('CV macro-F1', ascending=False).reset_index(drop=True)
tuning_df

#### Full Comparison — Selecting the Winner

In [ ]:
comparison_rows = []
for name, pipe in best_estimators.items():
    train_f1 = f1_score(y_train, pipe.predict(X_train), average='macro')
    test_f1 = f1_score(y_test, pipe.predict(X_test), average='macro')
    test_acc = accuracy_score(y_test, pipe.predict(X_test))
    cv_f1 = tuning_df.loc[tuning_df['Model'] == name, 'CV macro-F1'].values[0]
    comparison_rows.append({
        'Model': name, 'CV macro-F1': cv_f1, 'Test macro-F1': test_f1,
        'Test Accuracy': test_acc, 'Train macro-F1': train_f1, 'Gap': train_f1 - test_f1,
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('CV macro-F1', ascending=False).reset_index(drop=True).round(4)
comparison_df

**Draft notes — rewrite in your own words for the video:**

- The winner, selected purely by CV macro-F1 (not by looking at the test column), is whichever model tops this table's `CV macro-F1` column — check your own run's exact top row.
- If a different model happens to score marginally higher on `Test macro-F1` than the CV-selected winner, that's expected sampling variance from a single 20% split — not a reason to switch. Selecting on the test set would mean the test set stops being a fair, independent check.
- After tuning, the train-test gaps for the tree-based ensembles shrink substantially compared to their untuned baseline — direct evidence the tuning step did its job of controlling overfitting.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for ax, (name, pipe) in zip(axes, best_estimators.items()):
    preds = pipe.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Low', 'High']).plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=11, fontweight='bold')
for ax in axes[len(best_estimators):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

**Draft notes — rewrite in your own words for the video:**

- Comparing all seven confusion matrices side by side (not just the winner's) shows the error *pattern*, not just the summary metric — e.g. which models lean toward over-predicting "High" (more false positives) versus under-predicting it (more false negatives).

#### Winner Deep-Dive: Learning Curve

In [ ]:
winner_name = comparison_df.iloc[0]['Model']
winner_pipe = best_estimators[winner_name]

print(f"Winner (by CV macro-F1): {winner_name}")
preds = winner_pipe.predict(X_test)
print(classification_report(y_test, preds, target_names=['Low', 'High']))

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    winner_pipe, X_train, y_train, cv=cv, scoring='f1_macro',
    train_sizes=np.linspace(0.2, 1.0, 6), n_jobs=-1, shuffle=True, random_state=SEED,
)
train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
val_mean, val_std = val_scores.mean(axis=1), val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)
ax.plot(train_sizes, train_mean, 'o-', label='Train')
ax.plot(train_sizes, val_mean, 'o-', label='CV')
ax.set_xlabel('Training examples')
ax.set_ylabel('Macro F1')
ax.set_title(f'Learning Curve — {winner_name}', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"Final train={train_mean[-1]:.3f}, CV={val_mean[-1]:.3f}, gap={train_mean[-1] - val_mean[-1]:.3f}")

**Draft notes — rewrite in your own words for the video:**

- If the train and CV lines converge as training set size increases (rather than staying far apart), that suggests the model isn't simply memorising — it's generalising, and more data would likely help further.
- If both lines plateau at a similar, middling score, that points to a ceiling driven by the *features* available rather than by model capacity — consistent with the Assignment 1 EDA finding that no single feature strongly predicts Fitness Level.

## Task 2.3 — Discussion of Results

This section covers: a results summary, a critical comparison of the three models, error analysis, business/domain implications, and methodological limitations with recommendations for future work.

This section covers: a results summary, a critical comparison of the seven models, error analysis, business/domain implications, and methodological limitations.

### Results Summary

In [ ]:
comparison_df

**Draft notes — rewrite in your own words for the video:**

- Across all seven tuned models, CV macro-F1 clusters fairly tightly in the low-to-mid 0.70s, with Logistic Regression, SVM (RBF), Hist Gradient Boosting, and Naive Bayes all close together — no single model dominates by a wide margin.
- KNN and (untuned) tree ensembles were the weakest performers before tuning; tuning narrowed but did not eliminate this gap.
- Test accuracy sits close to the ~75% majority-class baseline for several models, which is exactly why macro-F1 (not accuracy) is the metric used for ranking — accuracy alone would make an unhelpful "always predict Low" strategy look artificially competitive.

### Critical Comparison: Which Model Is Best?

In [ ]:
print(f"Winner (by CV macro-F1): {winner_name}")
print(f"CV macro-F1: {comparison_df.iloc[0]['CV macro-F1']:.4f}")
print(f"Test macro-F1: {comparison_df.iloc[0]['Test macro-F1']:.4f}")
print()
print("Closest rivals:")
comparison_df.iloc[1:3]

**Draft notes — rewrite in your own words for the video:**

- The winner is selected purely by CV macro-F1 — this is the stable, cross-validated estimate, not a single lucky/unlucky test split.
- If a rival model (e.g. one of the closest two shown above) happens to score marginally higher on the test column, that's expected variance from one 20% split, not grounds to switch — switching after seeing the test result would mean the test set is no longer an independent check on the decision.
- The margin between the top few models is narrow — worth stating explicitly that this is a *close* comparison, not a landslide, which is itself an honest reflection of how weak the underlying feature signal was in the Assignment 1 EDA (no correlation above ±0.24).
- Tuning mattered more for some models than others: the baseline table showed Random Forest and Hist Gradient Boosting overfitting badly untuned (train F1 ≈ 1.0); after tuning, their gap to the other models narrows considerably, though they still don't overtake the simpler models here — evidence that, for a dataset this size (~1,600 rows) with weak signal, model complexity isn't buying much.

### What's Driving the Predictions?

In [ ]:
# Compare feature importance (Random Forest) against coefficients (Logistic Regression),
# regardless of which model won overall — both give a view on what the data is telling every model.
rf_pipe = best_estimators['Random Forest']
logreg_pipe = best_estimators['Logistic Regression']

importances = rf_pipe.named_steps['clf'].feature_importances_
imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False)

coefs = logreg_pipe.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': coefs}).sort_values('coefficient', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(imp_df['feature'], imp_df['importance'], color='#4C9A8F')
axes[0].set_title('Random Forest — Feature Importance', fontweight='bold')
axes[0].invert_yaxis()

colors = ['#B3473C' if v < 0 else '#2E5A6B' for v in coef_df['coefficient']]
axes[1].barh(coef_df['feature'], coef_df['coefficient'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Logistic Regression — Coefficients', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

**Draft notes — rewrite in your own words for the video:**

- Both models agree that **Nutrition Quality and Activity Index are the strongest predictors** — consistent with the correlations found in the Assignment 1 EDA (+0.24 and +0.23 respectively). Worth stating this explicitly: the modelling results corroborate the earlier EDA rather than contradicting it.
- **Smoker status is surprisingly influential in both models** — worth connecting back to the counter-intuitive EDA finding that smokers had *higher* average Nutrition Quality scores. A univariate relationship (smoking vs nutrition) doesn't have to match the multivariate one (smoking vs fitness once everything else is accounted for) — a good example of why relying on individual correlations alone can mislead.

### Error Analysis: Where and Why Did the Model Fail?

In [ ]:
X_test_labelled = X_test.copy()
X_test_labelled['actual'] = y_test.values
X_test_labelled['predicted'] = winner_pipe.predict(X_test)

false_negatives = X_test_labelled[(X_test_labelled['actual'] == 1) & (X_test_labelled['predicted'] == 0)]
true_positives = X_test_labelled[(X_test_labelled['actual'] == 1) & (X_test_labelled['predicted'] == 1)]

error_comparison = pd.DataFrame({
    'False Negatives (missed "High")': false_negatives[feature_cols].mean(),
    'True Positives (correctly caught)': true_positives[feature_cols].mean(),
}).round(2)
error_comparison['Difference'] = (error_comparison['False Negatives (missed "High")'] - error_comparison['True Positives (correctly caught)']).round(2)
error_comparison.sort_values('Difference', key=abs, ascending=False)

**Draft notes — rewrite in your own words for the video:**

- Compare the genuinely "High" fitness people the winning model **missed** against the ones it **correctly caught** — look for which features differ most between the two groups.
- If (as found in an earlier version of this analysis) the missed cases have notably lower Nutrition Quality and a higher smoker rate than the correctly-caught cases, that indicates the model's errors aren't random — they concentrate on people who are genuinely fit but whose overall profile statistically resembles a typical "Low" fitness person. That's a substantive, model-specific finding, not a generic "the model isn't perfect" statement.

### Business / Domain Implications

**Draft notes — rewrite in your own words for the video:**

- Even with seven models, broader tuning, and SMOTE, macro-F1 tops out in the low-to-mid 0.70s — reinforcing what the Assignment 1 EDA already suggested: no individual feature strongly predicts Fitness Level, and clustering showed weak natural separability between the two classes. This performance ceiling looks driven by the data available, not by insufficient modelling effort.
- For a real deployment (e.g. a wellness app flagging users for a fitness programme), this means predictions should support human judgement rather than replace it — especially given the error analysis shows the model is least reliable on exactly the interesting edge cases (e.g. fit smokers with poor nutrition scores) a business might most want to catch.
- The choice between the top few close models in practice would come down to which type of error costs more in that specific business context — missing a genuinely fit person, or wrongly flagging someone who isn't.

### Methodological Limitations and Recommended Improvements

**Draft notes — rewrite in your own words for the video:**

- **Dataset size and class imbalance**: ~1,600 rows with a 75/25 split leaves relatively few genuine "High" examples (~320) even before any oversampling. SMOTE helps the training process see a balanced signal, but it synthesises new points rather than collecting new genuine data — a real increase in labelled "High" examples would likely help more than any modelling change.
- **Imputation risk carried forward from Assignment 1**: Activity Index was ~51% imputed, and it's one of the two most important features across models — meaning a meaningful share of the model's reliance sits on estimated, not observed, values.
- **Search scope**: `RandomizedSearchCV` with up to 12 draws per model is reasonable but not exhaustive, particularly for larger grids (Random Forest, Hist Gradient Boosting, MLP) — a wider search or more iterations could be explored with more compute time.
- **Only one train/test split evaluated at the end**: while cross-validation was used throughout tuning and selection, the final reported test-set numbers come from a single 80/20 split — repeating this with multiple random seeds would give a more robust final estimate and a sense of variance in the reported test metrics.
- **Recommended next step**: given the error analysis finding, a worthwhile follow-up would be testing whether adding an interaction term (e.g. Smoker × Nutrition Quality) helps the model better capture cases where the two features point in different directions.

## AI Declaration and Reflection

### AI Tools Declaration

TODO: Add which AI tools/services/models were used and for what

### AI Tools Reflection and Lessons Learnt

TODO: Add a short reflection on your experiences with the AI tools you used (and tools you may have discarded early on), emphasising lessons learnt and key usage patterns that was successful for you.

### AI Academic Integrity

TODO: Add a short statement about how you have ensured academic integrity in this assignment, especially in terms of AI usage. Expected 1 paragraph only.